In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = "/content/drive/MyDrive/法令遵循處/重大裁罰、非重大裁罰、法規修訂/法規"


In [ ]:
!pip install pymupdf python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.5 MB/s eta 0:00:00


In [ ]:
import os
import json
import fitz  # PyMuPDF
import docx  # python-docx
from google.colab import drive

# 1. 掛載 Google 雲端硬碟
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:


import os
import json
import fitz  # PyMuPDF
import docx  # python-docx
from google.colab import drive

# 1. 掛載 Google 雲端硬碟
drive.mount('/content/drive')

# ================= 參數設定區 =================
# 請將 FOLDER_PATH 設定為你的「法規」母資料夾路徑
FOLDER_PATH = '/content/drive/MyDrive/法令遵循處/重大裁罰、非重大裁罰、法規修訂/裁罰'
OUTPUT_JSON_PATH = '/content/drive/MyDrive/外規資料處理-財閥/output_all.json'
# ============================================

def read_txt(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()

def read_pdf(file_path):
    doc = fitz.open(file_path)
    text = "\n".join([page.get_text("text") for page in doc])
    doc.close()
    return text

def read_docx(file_path):
    doc = docx.Document(file_path)
    text = "\n".join([para.text for para in doc.paragraphs])
    return text

def main():
    all_data = []
    processed_count = 0
    error_count = 0

    if not os.path.exists(FOLDER_PATH):
        print(f"找不到資料夾：{FOLDER_PATH}，請確認路徑是否正確。")
        return

    print("開始批次處理檔案...")

    for root, dirs, files in os.walk(FOLDER_PATH):
        for file in files:
            file_path = os.path.join(root, file)
            ext = file.lower().split('.')[-1]

            # --- 核心邏輯：從路徑與檔名萃取資訊 ---

            # 1. 解析資料夾結構 (rel_path 會像是 "勞動部/2026-2-18")
            rel_path = os.path.relpath(root, FOLDER_PATH)
            parts = rel_path.split(os.sep)

            if rel_path == '.':
                category = "未分類"
                time_str = "未知時間"
            else:
                # 第一層資料夾是類別 (例如：勞動部)
                category = parts[0]
                # 第二層資料夾是時間 (例如：2026-2-18)，如果沒有第二層就給預設值
                time_str = parts[1] if len(parts) > 1 else "未知時間"

            # 2. 標題直接取「檔案名稱」，並去除副檔名
            title = os.path.splitext(file)[0]

            # --------------------------------------

            raw_text = ""
            try:
                # 讀取檔案內容
                if ext == 'txt':
                    raw_text = read_txt(file_path)
                elif ext == 'pdf':
                    raw_text = read_pdf(file_path)
                elif ext == 'docx':
                    raw_text = read_docx(file_path)
                elif ext == 'doc':
                    print(f"略過 {file}：請先將 .doc 轉為 .docx")
                    continue
                else:
                    continue

                # 整理內文 (去除多餘的空白行，保留有效換行)
                lines = [line.strip() for line in raw_text.split('\n') if line.strip()]

                # 防呆：如果內文第一行剛好跟標題(檔名)一樣或包含 ""，就把它刪掉，避免內文重複出現標題
                if lines and (title in lines[0] or ""):
                    lines = lines[1:]

                content = "\n".join(lines)

                # 組裝最終的 JSON 結構
                parsed_data = {
                    "資料類別": category,
                    "標題": title,
                    "時間": time_str,
                    "內文": content
                }

                all_data.append(parsed_data)
                processed_count += 1

            except Exception as e:
                print(f"錯誤 {file}：讀取或解析失敗 ({e})")
                error_count += 1

        with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as jf:
          for data in all_data:
            jf.write(json.dumps(data, ensure_ascii=False, separators=(',', ':')) + '\n')

    print("-" * 30)
    print(f"處理完成！共成功解析 {processed_count} 個檔案，失敗 {error_count} 個。")
    print(f"結果已儲存至：{OUTPUT_JSON_PATH}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
開始批次處理檔案...
------------------------------
處理完成！共成功解析 1117 個檔案，失敗 0 個。
結果已儲存至：/content/drive/MyDrive/外規資料處理-財閥/output_all.json


In [ ]:
import os
import json
import fitz  # PyMuPDF
import docx  # python-docx
from google.colab import drive

# 1. 掛載 Google 雲端硬碟
drive.mount('/content/drive')

# ================= 參數設定區 =================
# 請將 FOLDER_PATH 設定為你的「裁罰」母資料夾路徑
FOLDER_PATH = '/content/drive/MyDrive/法令遵循處/重大裁罰、非重大裁罰、法規修訂/裁罰'
OUTPUT_JSON_PATH = '/content/drive/MyDrive/外規資料處理-財閥/output_penalty.json'
# ============================================

def read_txt(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()

def read_pdf(file_path):
    doc = fitz.open(file_path)
    text = "\n".join([page.get_text("text") for page in doc])
    doc.close()
    return text

def read_docx(file_path):
    doc = docx.Document(file_path)
    text = "\n".join([para.text for para in doc.paragraphs])
    return text

# 新增了 institution_name (機構名稱) 參數
def parse_content(raw_text, category_name, institution_name):
    """將純文字拆解，並加入資料類別與機構名稱"""
    lines = [line.strip() for line in raw_text.split('\n') if line.strip()]

    if len(lines) < 3:
        return None

    title = lines[0].replace("", "").strip()
    time = lines[1].strip()

    # 內文保留換行符號 (\n)
    content = "\n".join(lines[2:])

    # 按照你的需求加入 key
    return {
        "資料類別": category_name,
        "機構名稱": institution_name,
        "標題": title,
        "時間": time,
        "內文": content
    }

def main():
    all_data = []
    processed_count = 0
    error_count = 0

    if not os.path.exists(FOLDER_PATH):
        print(f"找不到資料夾：{FOLDER_PATH}，請確認路徑是否正確。")
        return

    print("開始批次處理檔案...")

    for root, dirs, files in os.walk(FOLDER_PATH):
        for file in files:
            file_path = os.path.join(root, file)
            ext = file.lower().split('.')[-1]

            # --- 核心邏輯：判斷兩層資料夾名稱 ---
            # 取得相對於 FOLDER_PATH 的路徑 (例如: "重大裁罰/金管會" 或 "非重大裁罰/銀行局")
            rel_path = os.path.relpath(root, FOLDER_PATH)

            # 切割路徑
            parts = rel_path.split(os.sep)

            # 處理各種存放層級的情況
            if rel_path == '.':
                category = "未分類"
                institution = "未分類"
            elif len(parts) == 1:
                # 檔案直接放在「重大裁罰」底下，沒有放進機構資料夾
                category = parts[0]
                institution = "未分類"
            else:
                # 正常情況：有兩層以上的結構
                category = parts[0]    # 第一層：重大裁罰 / 非重大裁罰
                institution = parts[1] # 第二層：金管會 / 證期局 / 銀行局...
            # ------------------------------------

            raw_text = ""
            try:
                if ext == 'txt':
                    raw_text = read_txt(file_path)
                elif ext == 'pdf':
                    raw_text = read_pdf(file_path)
                elif ext == 'docx':
                    raw_text = read_docx(file_path)
                elif ext == 'doc':
                    print(f"略過 {file}：請先將 .doc 轉為 .docx")
                    continue
                else:
                    continue

                # 將兩個類別變數傳入解析函式
                parsed_data = parse_content(raw_text, category, institution)
                if parsed_data:
                    all_data.append(parsed_data)
                    processed_count += 1
                else:
                    print(f"警告 {file}：內容不足 3 行，無法解析結構。")

            except Exception as e:
                print(f"錯誤 {file}：讀取或解析失敗 ({e})")
                error_count += 1

    with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as jf:
          for data in all_data:
            jf.write(json.dumps(data, ensure_ascii=False, separators=(',', ':')) + '\n')

    print("-" * 30)
    print(f"處理完成！共成功解析 {processed_count} 個檔案，失敗 {error_count} 個。")
    print(f"結果已儲存至：{OUTPUT_JSON_PATH}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
開始批次處理檔案...
警告 前台証綜合證券股份有限公司及日盛證券股份有限公司受僱人於辦理通嘉科技股份有限公司股票初次上市承銷案件，有配合發行公司辦理配售、未以公平、合理方式辦理配售之情事..._2012-01-20.txt：內容不足 3 行，無法解析結構。
------------------------------
處理完成！共成功解析 1116 個檔案，失敗 0 個。
結果已儲存至：/content/drive/MyDrive/外規資料處理-財閥/output_penalty.json


In [ ]:
import os
import json

# ================= 參數設定區 =================
# 1. 請將這裡換成你單一檔案的「實際路徑」
INPUT_FILE_PATH = '/content/drive/MyDrive/法令遵循處/主管法規資料集/銀行局主管法規資料集.jsonl'

# 2. 設定處理完後，新檔案要存去哪裡 (包含新檔名)
OUTPUT_FILE_PATH = '/content/drive/MyDrive/新_主管機關/處理後_銀行局主管法規資料集.jsonl'

# 3. 直接在這裡設定你要加入的機構名稱
INSTITUTION_NAME = '銀行局'
# ============================================

def process_single_jsonl(input_path, output_path, inst_name):
    # 檢查檔案是否存在
    if not os.path.exists(input_path):
        print(f"❌ 找不到檔案：{input_path}，請確認路徑是否正確。")
        return

    print(f"開始處理檔案：{input_path}")
    line_count = 0

    try:
        # 開啟原檔案讀取，並開啟新檔案寫入
        with open(input_path, 'r', encoding='utf-8') as infile, \
             open(output_path, 'w', encoding='utf-8') as outfile:

            for line in infile:
                line = line.strip()
                if not line:
                    continue  # 跳過空白行

                try:
                    # 讀取原本的 JSON
                    original_data = json.loads(line)

                    # 建立新字典，確保「機構名稱」在第一個位置
                    new_data = {"機構名稱": inst_name}
                    new_data.update(original_data)

                    # 將新的字典轉回 JSONL 格式的字串，並寫入新檔
                    outfile.write(json.dumps(new_data, ensure_ascii=False) + '\n')
                    line_count += 1

                except json.JSONDecodeError as e:
                    print(f"⚠️ 解析錯誤 (在第 {line_count + 1} 行左右): {e}")

        print("-" * 30)
        print(f"✅ 處理完成！已成功為 {line_count} 筆資料加入 [機構名稱: {inst_name}]。")
        print(f"📂 新檔案已儲存至：{output_path}")

    except Exception as e:
        print(f"❌ 發生錯誤：{e}")

# 執行主程式
process_single_jsonl(INPUT_FILE_PATH, OUTPUT_FILE_PATH, INSTITUTION_NAME)

開始處理檔案：/content/drive/MyDrive/法令遵循處/主管法規資料集/銀行局主管法規資料集.jsonl
------------------------------
✅ 處理完成！已成功為 1617 筆資料加入 [機構名稱: 銀行局]。
📂 新檔案已儲存至：/content/drive/MyDrive/新_主管機關/處理後_銀行局主管法規資料集.jsonl


In [ ]:
!apt-get update
!apt-get install -y libreoffice

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,989 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,311 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages 

In [ ]:
import os
import json
import fitz  # PyMuPDF
import docx  # python-docx
import subprocess # 用來呼叫系統指令 (LibreOffice)
from google.colab import drive

# 1. 掛載 Google 雲端硬碟
drive.mount('/content/drive')

# ================= 參數設定區 =================
FOLDER_PATH = '/content/drive/MyDrive/金控規章辦法資料夾'
OUTPUT_JSON_PATH = '/content/drive/MyDrive/內規/內規彙整資料.json'
# ============================================

def convert_doc_to_docx(doc_path):
    """使用背景的 LibreOffice 將 .doc 轉為 .docx，並存在暫存區"""
    out_dir = '/tmp/converted_docs'
    os.makedirs(out_dir, exist_ok=True)

    # 呼叫 LibreOffice 進行無介面轉檔
    command = ['soffice', '--headless', '--convert-to', 'docx', '--outdir', out_dir, doc_path]
    subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # 組合出新檔案的路徑
    base_name = os.path.basename(doc_path)
    # 將附檔名 .doc 替換為 .docx
    docx_name = base_name[:-4] + '.docx'
    new_docx_path = os.path.join(out_dir, docx_name)

    # 確認檔案是否成功產生
    if os.path.exists(new_docx_path):
        return new_docx_path
    else:
        raise Exception("轉檔失敗，未能產生 .docx 檔案")

def read_txt(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read().strip()

def read_pdf(file_path):
    doc = fitz.open(file_path)
    text = "\n".join([page.get_text("text").strip() for page in doc])
    doc.close()
    return text

def read_docx(file_path):
    doc = docx.Document(file_path)
    full_text = []

    for para in doc.paragraphs:
        if para.text.strip():
            full_text.append(para.text.strip())

    for table in doc.tables:
        for row in table.rows:
            row_data = [cell.text.replace('\n', ' ').strip() for cell in row.cells if cell.text.strip()]
            if row_data:
                full_text.append(" | ".join(row_data))

    return "\n".join(full_text)

def main():
    all_data = []
    processed_count = 0
    error_count = 0

    if not os.path.exists(FOLDER_PATH):
        print(f"❌ 找不到資料夾：{FOLDER_PATH}")
        return

    print("開始解析內規資料夾...")

    for root, dirs, files in os.walk(FOLDER_PATH):
        for file in files:
            file_path = os.path.join(root, file)
            ext = file.lower().split('.')[-1]

            rel_path = os.path.relpath(root, FOLDER_PATH)
            parts = rel_path.split(os.sep)

            if rel_path == '.':
                department = "未分類處室"
                regulation = "未分類規章"
            elif len(parts) == 1:
                department = parts[0]
                regulation = "未分類規章"
            else:
                department = parts[0]
                regulation = parts[1]

            title = os.path.splitext(file)[0]
            raw_text = ""

            try:
                if ext == 'txt':
                    raw_text = read_txt(file_path)
                elif ext == 'pdf':
                    raw_text = read_pdf(file_path)
                elif ext == 'docx':
                    raw_text = read_docx(file_path)
                elif ext == 'doc':
                    # ======= 自動轉檔處理區 =======
                    print(f"🔄 正在將 {file} 轉為 .docx ...")
                    temp_docx_path = convert_doc_to_docx(file_path)
                    raw_text = read_docx(temp_docx_path)
                    # ==============================
                else:
                    continue

                if raw_text.strip():
                    all_data.append({
                        "處室": department,
                        "規章名稱": regulation,
                        "標題": title,
                        "內文": raw_text
                    })
                    processed_count += 1
                else:
                    print(f"⚠️ 警告 {file}：檔案為空或無法提取文字。")

            except Exception as e:
                print(f"❌ 錯誤 {file}：讀取或轉檔失敗 ({e})")
                error_count += 1

    with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as jf:
          for data in all_data:
            jf.write(json.dumps(data, ensure_ascii=False, separators=(',', ':')) + '\n')

    print("-" * 40)
    print(f"✅ 處理完成！成功解析 {processed_count} 個檔案，失敗 {error_count} 個。")
    print(f"📂 輸出結果已儲存至：{OUTPUT_JSON_PATH}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
開始解析內規資料夾...
🔄 正在將 職業安全衛生管理要點113.08.doc 轉為 .docx ...
🔄 正在將 出納作業管理要點113.11.doc 轉為 .docx ...
🔄 正在將 永續發展守則-對照表.doc 轉為 .docx ...
🔄 正在將 永續發展守則114.06.doc 轉為 .docx ...
🔄 正在將 財產暨物品管理須知114.10.doc 轉為 .docx ...
🔄 正在將 環境管理作業要點附件一覽表.doc 轉為 .docx ...
🔄 正在將 環境管理作業要點 (ISO 14001)113.08.doc 轉為 .docx ...
🔄 正在將 永續暨誠信經營委員會組織規程-對照表.doc 轉為 .docx ...
🔄 正在將 安全衛生工作守則113.08.doc 轉為 .docx ...
🔄 正在將 安全衛生工作守則附件一.doc 轉為 .docx ...
🔄 正在將 環境管理作業須知附件九：環境管理行動方案規劃表-ISO 14001(空白).doc 轉為 .docx ...
🔄 正在將 環境管理作業須知附件十二：環境溝通紀錄表(空白).doc 轉為 .docx ...
🔄 正在將 環境管理作業須知附件十：環境管理行動方案時程表(空白).doc 轉為 .docx ...
🔄 正在將 環境管理作業須知附件十四：內部稽查計畫(空白).doc 轉為 .docx ...
🔄 正在將 環境管理作業須知附件十一：環境管理行動方案結案報告表(空白).doc 轉為 .docx ...
🔄 正在將 文書處理要點-附件.doc 轉為 .docx ...
🔄 正在將 文書處理要點114.03.doc 轉為 .docx ...
🔄 正在將 附件六、國長大樓10樓辦公區門禁權限申請表.doc 轉為 .docx ...
🔄 正在將 附件七、門禁卡（識別證、來賓證、臨時卡）補、換發申請表.doc 轉為 .docx ...
🔄 正在將 附件二、臨時卡申請表.doc 轉為 .docx ...
🔄 正在將 附件三